# Train and evaluate in one hosted Colab GPU VM

CARLA and the PPO client both run inside this Colab VM. `/content` is fast temporary storage; Google Drive stores the validated CARLA archive and experiment artifacts. The first session downloads, validates, and caches CARLA 0.9.16. Later sessions restore that archive from Drive and extract it locally—CARLA itself never runs from Drive. No separate Linux machine, desktop, VNC, or remote tunnel is used.

**Before running cells:** choose **Runtime → Change runtime type → Runtime version 2026.04** when available and **Hardware accelerator → GPU**, then reconnect.

## 0 — User settings

In [ ]:
REPO_URL = "https://github.com/djdhillxn/carretera"
REPO_BRANCH = "main"
REPO_DIR = "/content/carretera"

DRIVE_ROOT = "/content/drive/MyDrive/CARLA_Highway_RL"
CARLA_SERVER_MODE = "managed"
CARLA_HOST = "127.0.0.1"
CARLA_PORT = 2000
CARLA_TM_PORT = 8000
CARLA_ROOT = "/content/CARLA_0.9.16"
CARLA_ARCHIVE_URL = "https://tiny.carla.org/carla-0-9-16-linux"
CARLA_ARCHIVE_LOCAL = "/content/CARLA_0.9.16.tar.gz"
CARLA_ARCHIVE_DRIVE = "/content/drive/MyDrive/CARLA_Highway_RL/runtime/CARLA_0.9.16.tar.gz"
CARLA_CACHE_DIR = "/content/carla_cache"

RUN_NAME = "ppo_seed_0"
SEED = 0
TARGET_TOTAL_TIMESTEPS = 50000
TRAINING_CHUNK_TIMESTEPS = 5000
TRAINING_CHUNKS_THIS_SESSION = 1  # Keep one chunk as the default action.
EVALUATION_SLICES_THIS_SESSION = 1
EVALUATION_SLICE_SIZE = 15
RUN_TINY_SANITY = False

## 1 — Common initialization

If the runtime selection above changed, reconnect before proceeding. These cells are safe to rerun after a disconnect.

In [ ]:
import os, platform, shutil, subprocess, sys, zipfile, shlex
from pathlib import Path

class CompletedProcessInfo:
    def __init__(self, command, returncode, stdout):
        self.args = command
        self.returncode = returncode
        self.stdout = stdout

def run_cmd(cmd, **kwargs):
    cmd_str = [str(x) for x in cmd]
    print("+", shlex.join(cmd_str))
    kwargs.pop("capture_output", None)
    kwargs.pop("check", None)
    kwargs.pop("text", None)
    process = subprocess.Popen(cmd_str, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, **kwargs)
    output_lines = []
    while True:
        line = process.stdout.readline()
        if not line and process.poll() is not None:
            break
        if line:
            sys.stdout.write(line)
            sys.stdout.flush()
            output_lines.append(line)
    process.wait()
    combined_output = "".join(output_lines)
    if process.returncode != 0:
        raise RuntimeError("Command '%s' failed with exit code %d" % (shlex.join(cmd_str), process.returncode))
    return CompletedProcessInfo(cmd_str, process.returncode, combined_output)

print("Python:", sys.version)
print("OS/architecture:", platform.platform(), platform.machine())
print("Disk:", shutil.disk_usage("/content") if Path("/content").exists() else shutil.disk_usage("/"))
subprocess.run(["nvidia-smi"], check=False)


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)

In [ ]:
repo = Path(REPO_DIR)
if not repo.exists():
    run_cmd(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR])
else:
    status = run_cmd(["git", "status", "--short"], cwd=repo, capture_output=True, text=True)
    print(status.stdout or "Working tree is clean.")
    run_cmd(["git", "fetch", "origin", REPO_BRANCH], cwd=repo)
    if not status.stdout.strip():
        run_cmd(["git", "merge", "--ff-only", "origin/" + REPO_BRANCH], cwd=repo)
    else:
        print("Local changes detected; skipped update without resetting or cleaning.")
os.chdir(REPO_DIR)
for expected in ("run.py", "carla_env.py", "policies.py", "config.yaml", "requirements.txt"):
    assert Path(expected).is_file(), expected
print("Repository:", Path.cwd())

Install only CARLA's required Linux utilities and libraries. Colab supplies the NVIDIA/CUDA driver; this does not install a desktop or a replacement driver.

In [ ]:
run_cmd(["apt-get", "update", "-qq"])
run_cmd(["apt-get", "install", "-y", "-qq", "aria2", "libvulkan1", "vulkan-tools", "libomp5", "libx11-6", "libxext6", "libxrender1", "libsm6", "libglib2.0-0"])

Install ordinary Python requirements without replacing Colab's CUDA-enabled PyTorch. NumPy 2 is supported; there is no NumPy downgrade or automatic runtime restart. The final command verifies key imports in a fresh process, which avoids confusing already-imported modules with installed versions.

In [ ]:
run_cmd([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])
version_check = "import importlib.metadata as m; names=['numpy','gymnasium','stable-baselines3','pandas','matplotlib','PyYAML']; print({n:m.version(n) for n in names}); import numpy, gymnasium, stable_baselines3, pandas, matplotlib, yaml"
run_cmd([sys.executable, "-c", version_check])
print("Fresh-process imports passed. A runtime restart is not required.")

In [ ]:
overrides = {
    "CARLA_SERVER_MODE": CARLA_SERVER_MODE,
    "CARLA_HOST": CARLA_HOST,
    "CARLA_PORT": str(CARLA_PORT),
    "CARLA_TM_PORT": str(CARLA_TM_PORT),
    "CARLA_ROOT": CARLA_ROOT,
    "CARLA_ARCHIVE_URL": CARLA_ARCHIVE_URL,
    "CARLA_ARCHIVE_LOCAL": CARLA_ARCHIVE_LOCAL,
    "CARLA_ARCHIVE_DRIVE": CARLA_ARCHIVE_DRIVE,
    "CARLA_CACHE_DIR": CARLA_CACHE_DIR,
    "HIGHWAY_RL_ARTIFACT_ROOT": str(Path(REPO_DIR) / "artifacts"),
    "HIGHWAY_RL_DRIVE_ROOT": DRIVE_ROOT,
}
os.environ.update(overrides)
print(overrides)

In [ ]:
run_cmd([sys.executable, "run.py", "sync", "--config", "config.yaml", "--from-drive"])
run_cmd([sys.executable, "run.py", "validate-config", "--config", "config.yaml"])

## 2 — Provision CARLA 0.9.16

The first run downloads to local storage, validates the archive, caches it in Drive, extracts locally, and installs the matching packaged wheel. A fresh later VM restores and verifies the Drive cache. This is idempotent and does not start the server.

In [ ]:
run_cmd([sys.executable, "run.py", "runtime", "status", "--config", "config.yaml"])
run_cmd([sys.executable, "run.py", "runtime", "prepare", "--config", "config.yaml"])
run_cmd([sys.executable, "run.py", "runtime", "status", "--config", "config.yaml"])

import importlib.metadata, json
prepare = json.loads(Path("artifacts/logs/runtime/runtime_prepare_manifest.json").read_text())
status = json.loads(Path("artifacts/logs/runtime/runtime_status.json").read_text())
print({key: prepare.get(key) for key in ("package_root", "archive_source", "drive_cache_validated", "archive_sha256", "wheel")})
print("Free local GiB:", status["storage"]["local_free_gb"])
assert Path(CARLA_ROOT, "CarlaUE4.sh").is_file()
assert importlib.metadata.version("carla") == "0.9.16"

## 3 — GPU, Vulkan, server, and doctor

In [ ]:
run_cmd(["nvidia-smi"])
run_cmd(["vulkaninfo", "--summary"])
run_cmd([sys.executable, "run.py", "runtime", "status", "--config", "config.yaml", "--strict"])
run_cmd([sys.executable, "run.py", "server", "status", "--config", "config.yaml"])
runtime_status = json.loads(Path("artifacts/logs/runtime/runtime_status.json").read_text())
if not runtime_status["managed_server"]["record_state"]["active"]:
    run_cmd([sys.executable, "run.py", "server", "start", "--config", "config.yaml"])
run_cmd([sys.executable, "run.py", "doctor", "--config", "config.yaml"])
doctor = json.loads(Path("artifacts/logs/runtime/doctor.json").read_text())
print("Highway candidates:", doctor.get("highway_candidates"))

## 4 — Environment smoke tests

In [ ]:
run_cmd([sys.executable, "run.py", "offline-self-test", "--config", "config.yaml"])
run_cmd([sys.executable, "run.py", "smoke", "--config", "config.yaml"])
smoke = json.loads(Path("artifacts/logs/runtime/smoke_test.json").read_text())
print(json.dumps({key: smoke.get(key) for key in ("gymnasium_checker", "stable_baselines3_checker", "keep_lane_episode", "rule_based_episode", "fresh_environment_actor_count", "double_close")}, indent=2))

## 5 — Optional tiny PPO plumbing run

This 2,000-decision run is only a plumbing check, never the final model.

In [ ]:
if RUN_TINY_SANITY:
    tiny_name = "ppo_plumbing_seed_0"
    tiny_root = Path("artifacts/models") / tiny_name
    if tiny_root.exists() and any(tiny_root.iterdir()):
        print("Tiny run already exists; refusing to overwrite:", tiny_root)
    else:
        run_cmd([sys.executable, "run.py", "train", "--config", "config.yaml", "--run-name", tiny_name, "--seed", str(SEED), "--total-timesteps", "2000"])
        run_cmd([sys.executable, "run.py", "sync", "--config", "config.yaml", "--to-drive"])
else:
    print("RUN_TINY_SANITY is False; skipped.")

## 6 — Primary restart-safe training

The default action runs exactly one 5,000-decision chunk. Every chunk writes a checkpoint/final model, episode CSV, TensorBoard data, metadata, plots, and then synchronizes artifacts. PPO remains on CPU per `config.yaml`; CARLA uses the GPU.

In [ ]:
run_cmd([sys.executable, "run.py", "sync", "--config", "config.yaml", "--from-drive"])
run_root = Path("artifacts/models") / RUN_NAME
for chunk_number in range(TRAINING_CHUNKS_THIS_SESSION):
    checkpoints = sorted(path for path in (run_root / "checkpoints").glob("checkpoint_*.zip") if zipfile.is_zipfile(path)) if run_root.exists() else []
    latest = checkpoints[-1] if checkpoints else None
    completed = int(latest.stem.split("_")[-1]) if latest else 0
    additional = min(TRAINING_CHUNK_TIMESTEPS, TARGET_TOTAL_TIMESTEPS - completed)
    print({"completed": completed, "target": TARGET_TOTAL_TIMESTEPS, "checkpoint": str(latest) if latest else None, "proposed_additional": additional, "model_root": str(run_root), "training_csv": f"artifacts/logs/train/{RUN_NAME}_episodes.csv"})
    if additional <= 0:
        print("Target already reached.")
        break
    if latest:
        command = [sys.executable, "run.py", "train", "--config", "config.yaml", "--run-name", RUN_NAME, "--seed", str(SEED), "--resume", str(latest), "--additional-timesteps", str(additional)]
    else:
        if run_root.exists() and any(run_root.iterdir()):
            raise RuntimeError(f"Existing run has no valid checkpoint; refusing a fresh overwrite: {run_root}")
        command = [sys.executable, "run.py", "train", "--config", "config.yaml", "--run-name", RUN_NAME, "--seed", str(SEED), "--total-timesteps", str(additional)]
    run_cmd(command)
    run_cmd([sys.executable, "run.py", "sync", "--config", "config.yaml", "--to-drive"])
print("Run another session/cell execution for the next chunk; the default does not launch all 50K.")

In [ ]:
from IPython.display import display, Image
for plot in ("artifacts/plots/training_episode_return.png", "artifacts/plots/training_success_collision.png"):
    if Path(plot).exists():
        display(Image(filename=plot))

## 7 — Paired evaluation manifest

In [ ]:
manifest_path = Path("artifacts/manifests/evaluation_manifest.json")
if not manifest_path.exists():
    run_cmd([sys.executable, "run.py", "make-eval-manifest", "--config", "config.yaml"])
manifest = json.loads(manifest_path.read_text())
rows = manifest["scenarios"]
print({"conditions": len(rows), "densities": len({r['traffic_density'] for r in rows}), "weather": len({r['weather'] for r in rows}), "seeds": len({r['seed'] for r in rows}), "hash": manifest['manifest_hash'], "path": str(manifest_path)})
run_cmd([sys.executable, "run.py", "sync", "--config", "config.yaml", "--to-drive"])

## 8 — Quick paired evaluation

In [ ]:
import pandas as pd
model_path = Path("artifacts/models") / RUN_NAME / "final_model.zip"
assert model_path.is_file(), f"Missing trained model: {model_path}"
run_cmd([sys.executable, "run.py", "evaluate", "--config", "config.yaml", "--manifest", str(manifest_path), "--model", str(model_path), "--quick", "--resume-existing"])
run_cmd([sys.executable, "run.py", "sync", "--config", "config.yaml", "--to-drive"])
episodes = pd.read_csv("artifacts/evaluations/episode_results.csv")
display(episodes.tail(12))
display(episodes[(episodes.success == 0) | (episodes.collision == 1)].tail(12))

## 9 — Full restartable evaluation

Each default execution evaluates at most one 15-condition slice, writes each episode immediately, resumes completed policy/condition pairs, and syncs afterward.

In [ ]:
run_cmd([sys.executable, "run.py", "sync", "--config", "config.yaml", "--from-drive"])
policies = manifest and ["ppo", "random", "keep_lane", "rule_based"]
for _ in range(EVALUATION_SLICES_THIS_SESSION):
    output = Path("artifacts/evaluations/episode_results.csv")
    done = set()
    if output.exists():
        existing = pd.read_csv(output)
        if existing.duplicated(["policy", "condition_id"]).any():
            raise RuntimeError("Duplicate policy/condition pairs found; inspect before continuing.")
        done = set(zip(existing.policy, existing.condition_id))
    missing_indices = [i for i, row in enumerate(rows) if any((p, row['condition_id']) not in done for p in policies)]
    if not missing_indices:
        print("Full evaluation complete.")
        break
    start = missing_indices[0]
    end = min(start + EVALUATION_SLICE_SIZE, len(rows))
    run_cmd([sys.executable, "run.py", "evaluate", "--config", "config.yaml", "--manifest", str(manifest_path), "--model", str(model_path), "--start-index", str(start), "--end-index", str(end), "--resume-existing"])
    run_cmd([sys.executable, "run.py", "sync", "--config", "config.yaml", "--to-drive"])
    updated = pd.read_csv(output).drop_duplicates(["policy", "condition_id"])
    print(f"Completed {len(updated)} / {len(rows) * len(policies)} policy-condition pairs; {len(rows) * len(policies) - len(updated)} remain.")

## 10 — Analysis and report data

In [ ]:
episode_path = "artifacts/evaluations/episode_results.csv"
run_cmd([sys.executable, "run.py", "analyze", "--config", "config.yaml", "--episodes", episode_path])
run_cmd([sys.executable, "run.py", "report-data", "--config", "config.yaml"])
run_cmd([sys.executable, "run.py", "sync", "--config", "config.yaml", "--to-drive"])
display(pd.read_csv("artifacts/evaluations/summary_results.csv"))
display(pd.read_csv("artifacts/evaluations/paired_comparisons.csv"))
display(pd.read_csv("artifacts/evaluations/bootstrap_intervals.csv").head(20))
for plot in sorted(Path("artifacts/plots").glob("*.png")):
    display(Image(filename=str(plot)))

## 11 — Safe shutdown

In [ ]:
run_cmd([sys.executable, "run.py", "sync", "--config", "config.yaml", "--to-drive"])
if CARLA_SERVER_MODE == "managed":
    run_cmd([sys.executable, "run.py", "server", "stop", "--config", "config.yaml"])
else:
    print("External server mode: no server was stopped.")
print("Experiment artifacts:", DRIVE_ROOT)
print("CARLA archive cache:", CARLA_ARCHIVE_DRIVE)
print("The extracted local package was intentionally retained until this VM ends.")